In [1]:
!pip -q install transformers torch

import os
import re
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
model.eval()

print("Device:", device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device: cpu


In [3]:
poem = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

In [4]:
import re

LINE_RE = re.compile(r"^(.*?)([A-Za-z][A-Za-z'\-]*)([;:,.\?!]*)\s*$")

def split_last_word(line: str):
    m = LINE_RE.match(line)
    if not m:
        return None
    return m.group(1), m.group(2), m.group(3)

for i, line in enumerate(poem.splitlines(), 1):
    p = split_last_word(line)
    if p:
        prefix, last_word, punct = p
        print(f"{i:02d}: replace '{last_word}{punct}'")


01: replace 'winter'
02: replace 'boughs'
03: replace 'snow;'
04: replace 'time'
05: replace 'ice,'
06: replace 'glitter'
07: replace 'think'
08: replace 'wind,'
09: replace 'leaves,'
10: replace 'land'
11: replace 'wind'
12: replace 'place'
13: replace 'snow,'
14: replace 'beholds'
15: replace 'is.'


In [5]:
import torch

def looks_like_word(word: str) -> bool:
    # after stripping, must be a single "word-like" chunk
    if not word:
        return False
    if any(ch.isspace() for ch in word):
        return False
    # allow letters + optional apostrophes/hyphens
    if not re.fullmatch(r"[A-Za-z][A-Za-z'\-]*", word):
        return False
    return True

@torch.no_grad()
def pick_ranked_word(prefix: str, rank: int = 7, scan_limit: int = 10000) -> str:
    """
    Returns the rank-th most probable next WORD (approximation: single GPT-2 token
    that begins with a space). This avoids BPE fragments like 'idd'/'ich'.
    """
    # CRITICAL FIX: remove trailing spaces so next token includes leading space
    prefix = prefix.rstrip()

    input_ids = tokenizer.encode(prefix, return_tensors="pt").to(device)
    logits = model(input_ids).logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    sorted_ids = torch.argsort(probs, descending=True)

    words = []
    scanned = 0

    for tid in sorted_ids:
        scanned += 1
        if scanned > scan_limit:
            break

        tok = tokenizer.decode([tid.item()])

        # CRITICAL: only accept "new word" tokens
        if not tok.startswith(" "):
            continue

        cand = tok.strip()  # remove that leading space

        if looks_like_word(cand):
            words.append(cand)
            if len(words) >= rank:
                return words[rank - 1]

    return words[-1] if words else ""


In [6]:
@torch.no_grad()
def replace_last_word_line(line: str, rank: int) -> str:
    parts = split_last_word(line)
    if not parts:
        return line

    prefix, last_word, punct = parts
    replacement = pick_ranked_word(prefix, rank=rank)
    return f"{prefix}{replacement}{punct}"

def transform_poem(poem_text: str, rank: int) -> str:
    out_lines = []
    for line in poem_text.splitlines():
        if line.strip() == "":
            out_lines.append(line)
        else:
            out_lines.append(replace_last_word_line(line, rank))
    return "\n".join(out_lines)


In [7]:
lines = poem.splitlines()

for i in range(3):
    prefix, last_word, punct = split_last_word(lines[i])
    repl = pick_ranked_word(prefix, rank=7)
    print(f"Line {i+1}: {last_word} -> {repl}")


Line 1: winter -> her
Line 2: boughs -> death
Line 3: snow -> oil


In [8]:
p7 = transform_poem(poem, 7)
print(p7)

One must have a mind of her
To regard the frost and the death
Of the pine-trees crusted with oil;
And have been cold a long few
To behold the junipers shagged with white,
The spruces rough in the distant night
Of the January sun; and not to have
Of any misery in the sound of the sound,
In the sound of a few shots,
Which is the sound of the voice
Full of the same story
That is blowing in the same bare head
For the listener, who listens in the following,
And, nothing himself, I
Nothing that is not there and the nothing that isn.


In [14]:
p4800= transform_poem(poem, 4800)
print(p4800)

One must have a mind of forge
To regard the frost and the plethora
Of the pine-trees crusted with bastard;
And have been cold a long associate
To behold the junipers shagged with unpleasant,
The spruces rough in the distant termin
Of the January sun; and not to casual
Of any misery in the sound of the produce,
In the sound of a few vast,
Which is the sound of the spy
Full of the same organizational
That is blowing in the same bare Egyptian
For the listener, who listens in the layered,
And, nothing himself, Ry
Nothing that is not there and the nothing that clues.


In [15]:
import os
from datetime import datetime

A02_DIR = "A02"
os.makedirs(A02_DIR, exist_ok=True)

p7_path = os.path.join(A02_DIR, "P+7.txt")
px_val = 4800
px_path = os.path.join(A02_DIR, f"P+{px_val}.txt")

with open(p7_path, "w", encoding="utf-8") as f:
    f.write(p7.strip() + "\n")

with open(px_path, "w", encoding="utf-8") as f:
    f.write(p4800.strip() + "\n")

print("Saved:", p7_path)
print("Saved:", px_path)


Saved: A02/P+7.txt
Saved: A02/P+4800.txt


In [ ]:
def preview_poem(text, head=3, tail=4):
    lines = text.splitlines()
    return "\n".join(lines[:head] + ["..."] + lines[-tail:])

test_xs = [21, 25, 33, 44, 69]

for x in test_xs:
    px = transform_poem(poem, x)
    print("\n" + "="*40)
    print(f"P+{x} preview:")
    print(preview_poem(px, head=4, tail=5))



P+21 preview:
One must have a mind of order
To regard the frost and the dust
Of the pine-trees crusted with yellow;
And have been cold a long times
...
Full of the same substance
That is blowing in the same bare toe
For the listener, who listens in the next,
And, nothing himself, nor
Nothing that is not there and the nothing that they.

P+25 preview:
One must have a mind of mystery
To regard the frost and the loss
Of the pine-trees crusted with pollen;
And have been cold a long term
...
Full of the same idea
That is blowing in the same bare pocket
For the listener, who listens in the audience,
And, nothing himself, which
Nothing that is not there and the nothing that seems.

P+33 preview:
One must have a mind of yours
To regard the frost and the chill
Of the pine-trees crusted with blue;
And have been cold a long wait
...
Full of the same game
That is blowing in the same bare areas
For the listener, who listens in the original,
And, nothing himself, what
Nothing that is not there and 

In [ ]:
def last_words_only(poem_text, rank):
    out = []
    for line in poem_text.splitlines():
        parts = split_last_word(line)
        if not parts:
            continue
        prefix, last_word, punct = parts
        repl = pick_ranked_word(prefix, rank=rank)
        out.append(repl + punct)
    return out

test_xs = [4, 10, 13, 14, 17, 20, 36, 44, 68, 100]
for x in test_xs:
    lw = last_words_only(poem, x)
    print("\n" + "-"*40)
    print(f"P+{x} last-word list:")
    print(", ".join(lw))



----------------------------------------
P+4 last-word list:
your, ice, salt;, long, blood,, sky, say, world,, gunshots,, engine, type, air, first,, no, was.

----------------------------------------
P+10 last-word list:
one, storm, white;, year, tw,, shadows, my, earth,, loud,, music, year, face, most,, though, exists.

----------------------------------------
P+13 last-word list:
what, sun, their;, night, arrows,, forest, a, drums,, footsteps,, ship, and, water, default,, as, will.

----------------------------------------
P+14 last-word list:
it, water, leaves;, bit, feathers,, corners, all, war,, people,, fire, way, skin, form,, for, doesn.

----------------------------------------
P+17 last-word list:
how, sudden, dried;, hard, sharp,, desert, get, great,, lines,, waves, as, palm, current,, that, never.

----------------------------------------
P+20 last-word list:
self, darkness, his;, couple, two,, light, give, car,, moments,, phone, in, light, right,, only, happens.

---------